In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

In [ ]:
tools = [
    {
        "name": "extract_return_request",
        "description": "Extract a structured return request from a customer support message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The customer's order ID, or null if not provided."
                },
                "item": {
                    "type": ["string", "null"],
                    "description": "The item the customer wants to return or replace."
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "normal_return",
                        "damaged_item",
                        "billing_dispute",
                        "policy_exception",
                        "unclear",
                        "other"
                    ],
                    "description": "The main reason for the request."
                },
                "reason_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when reason is other or unclear."
                },
                "desired_action": {
                    "type": "string",
                    "enum": [
                        "refund",
                        "replacement",
                        "exchange",
                        "store_credit",
                        "unclear",
                        "other"
                    ]
                },
                "desired_action_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when desired_action is other or unclear."
                },
                "evidence_provided": {
                    "type": "boolean",
                    "description": "Whether the customer already provided evidence, such as a photo."
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "normal", "high", "unclear"]
                },
                "missing_information": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    },
                    "description": "Information needed before the request can be processed."
                }
            },
            "required": [
                "order_id",
                "item",
                "reason",
                "reason_detail",
                "desired_action",
                "desired_action_detail",
                "evidence_provided",
                "urgency",
                "missing_information"
            ]
        }
    }
]

In [9]:
customer_message = """
I got my shoes yesterday and they are scratched.
I want a replacement. I can send a photo if needed.
"""

message = client.messages.create(
    model=model,
    max_tokens=500,
    temperature=0,
    tools=tools,
    tool_choice={
        "type": "tool",
        "name": "extract_return_request"
    },
    messages=[
        {
            "role": "user",
            "content": f"""
Extract the return request from this customer message.

Use these normalization rules:
- If the customer did not provide an order ID, set order_id to null.
- If the customer says they can provide evidence later, evidence_provided is false.
- Use damaged_item only when the item arrived broken, scratched, defective, or unusable.
- Use unclear when the message does not contain enough information to choose confidently.

Customer message:
{customer_message}
"""
        }
    ]
)

tool_use = next(
    block for block in message.content
    if block.type == "tool_use"
)

structured_data = tool_use.input

print(structured_data)

{'order_id': None, 'item': 'shoes', 'reason': 'damaged_item', 'reason_detail': None, 'desired_action': 'replacement', 'desired_action_detail': None, 'evidence_provided': False, 'urgency': 'normal', 'missing_information': ['order_id']}


In [8]:
message = client.messages.create(
    model=model,
    max_tokens=500,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": """
Extract the return request from this customer message.
Return only valid JSON.

Customer message:
I got my shoes yesterday and they are scratched.
I want a replacement. I can send a photo if needed.
"""
        }
    ]
)

print(message.content[0].text)

```json
{
  "reason": "scratched",
  "resolution": "replacement",
  "evidence_available": true,
  "evidence_type": "photo",
  "product": "shoes"
}
```
